# NuFold — Google Colab Verification & Custom Target Notebook

**Project:** RNA 3D Structure Prediction for Template-Sparse RNA Sequences  
**Model:** NuFold  
**Official repository:** `kiharalab/NuFold`

This notebook contains:
1. a verified bundled-example workflow using `2DER_C`
2. a custom-target workflow using your own RNA sequence + predicted SS + uploaded/precomputed A3M

Verified bundled output: `test_output/2DER_C/2DER_C_rank_1.pdb`.

## 1. GPU and host check

In [ ]:
import os, sys, time, threading, subprocess, platform, re, csv, shutil
from pathlib import Path

print("Host Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone the official NuFold repository

In [ ]:
%cd /content
if not Path("/content/NuFold").exists():
    !git clone https://github.com/kiharalab/NuFold.git
else:
    print("NuFold repository already exists; reusing it.")

%cd /content/NuFold
commit = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("NuFold commit:", commit)

## 3. Install NuFold dependencies

The official README notes NuFold was trained with PyTorch `1.10.0+cu111` and tested with `2.0.1+cu117`. The verified Colab workflow succeeded with the Colab-provided GPU PyTorch stack.

In [ ]:
%cd /content/NuFold
!pip install -q ml-collections dm-tree deepspeed protobuf scipy biopython numpy matplotlib py3Dmol
print("Dependency installation complete.")

## 4. Verify Python / NumPy / PyTorch / CUDA

In [ ]:
import torch
import numpy as np

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA GPU is not available.")

## 5. Download the official NuFold checkpoint

Expected path: `checkpoints/global_step145245.pt`

In [ ]:
%cd /content/NuFold

checkpoint_dir = Path("/content/NuFold/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "global_step145245.pt"

if not checkpoint_path.exists():
    !wget -O /content/NuFold/checkpoints/global_step145245.pt http://kiharalab.org/nufold/global_step145245.pt
else:
    print("Checkpoint already exists.")

if not checkpoint_path.exists() or checkpoint_path.stat().st_size == 0:
    raise FileNotFoundError("NuFold checkpoint download failed.")

print("Checkpoint:", checkpoint_path)
print("Checkpoint size (MB):", round(checkpoint_path.stat().st_size/(1024**2),2))

## 6. Inspect the bundled verification target

Verified bundled files:

```text
test_input/2DER_C/
├── 2DER_C.fasta
├── 2DER_C.a3m
└── 2DER_C.single.ipknot.ss
```

In [ ]:
%cd /content/NuFold

target_dir = Path("/content/NuFold/test_input/2DER_C")

print("Bundled target files:")
for p in sorted(target_dir.glob("*")):
    print(p)

required = [
    target_dir/"2DER_C.fasta",
    target_dir/"2DER_C.a3m",
    target_dir/"2DER_C.single.ipknot.ss",
]
missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError("Missing bundled NuFold inputs: " + ", ".join(missing))

print("All required bundled inputs are present.")

## 7. Check NuFold command-line help

In [ ]:
%cd /content/NuFold
h = subprocess.run(
    ["python","run_nufold.py","-h"],
    cwd="/content/NuFold",
    capture_output=True,
    text=True
)
print(h.stdout)
print(h.stderr)
if h.returncode != 0:
    raise RuntimeError("Could not load run_nufold.py")

## 8. Run bundled NuFold verification + runtime / GPU-memory logging

In [ ]:
%cd /content/NuFold

output_dir = Path("/content/NuFold/test_output")
output_dir.mkdir(parents=True, exist_ok=True)

gpu_samples = []
stop_monitor = False

def monitor_gpu():
    global stop_monitor
    while not stop_monitor:
        try:
            out = subprocess.check_output(
                ["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if out:
                gpu_samples.append(int(out[0]))
        except Exception:
            pass
        time.sleep(0.5)

t = threading.Thread(target=monitor_gpu, daemon=True)
t.start()
start = time.perf_counter()

proc = subprocess.run([
    "python","run_nufold.py",
    "--ckpt_path","checkpoints/global_step145245.pt",
    "--input_fasta","test_input/2DER_C/2DER_C.fasta",
    "--input_dir","test_input/",
    "--output_dir","test_output",
    "--config_preset","initial_training"
], cwd="/content/NuFold", capture_output=True, text=True)

elapsed = time.perf_counter() - start
stop_monitor = True
t.join(timeout=2)
peak_gpu_mb = max(gpu_samples) if gpu_samples else None

print("===== STDOUT =====")
print(proc.stdout)
print("===== STDERR =====")
print(proc.stderr)
print("=== RUN SUMMARY ===")
print("Exit code:", proc.returncode)
print(f"Runtime: {elapsed:.2f} s")
print("Peak observed GPU memory:", f"{peak_gpu_mb} MiB" if peak_gpu_mb is not None else "Unavailable")

if proc.returncode != 0:
    raise RuntimeError("NuFold inference failed. Check STDERR above.")

## 9. Verify bundled prediction outputs

In [ ]:
expected_pdb = Path("/content/NuFold/test_output/2DER_C/2DER_C_rank_1.pdb")

print("Expected PDB:", expected_pdb)
print("Exists:", expected_pdb.exists())

for p in sorted(Path("/content/NuFold/test_output").rglob("*")):
    if p.is_file():
        print(p)

if not expected_pdb.exists():
    raise FileNotFoundError("Expected 2DER_C_rank_1.pdb was not generated.")

## 10. Save bundled-run reproducibility metadata

In [ ]:
gpu_name = subprocess.check_output(
    ["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
    text=True
).strip().splitlines()[0]

record = {
    "model":"NuFold",
    "model_commit":commit,
    "checkpoint":"global_step145245.pt",
    "target":"2DER_C",
    "input_fasta":"test_input/2DER_C/2DER_C.fasta",
    "input_msa":"test_input/2DER_C/2DER_C.a3m",
    "input_ss":"test_input/2DER_C/2DER_C.single.ipknot.ss",
    "output":str(expected_pdb),
    "python":sys.version.split()[0],
    "numpy":np.__version__,
    "pytorch":torch.__version__,
    "gpu":gpu_name,
    "runtime_seconds":round(elapsed,3),
    "peak_gpu_memory_mib_observed":peak_gpu_mb,
    "exit_code":proc.returncode
}

bundled_log_path = Path("/content/nufold_run_log.csv")
with bundled_log_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=record.keys())
    w.writeheader()
    w.writerow(record)

print(record)
print("Saved:", bundled_log_path)

## 11. Optional — save bundled verification outputs to Google Drive

In [ ]:
# OPTIONAL CELL
# from google.colab import drive
# drive.mount("/content/drive")
# destination = Path("/content/drive/MyDrive/RNA3D_Benchmark/NuFold")
# destination.mkdir(parents=True, exist_ok=True)
# shutil.copy2(expected_pdb, destination/expected_pdb.name)
# shutil.copy2(bundled_log_path, destination/bundled_log_path.name)
# print("Saved to:", destination)

# Custom Target Workflow

Standalone NuFold is not sequence-only. A custom target needs:

```text
TARGET.fasta
TARGET.a3m
TARGET.single.ipknot.ss
```

The cells below create FASTA, predict SS with IPknot, accept a precomputed A3M, run NuFold, verify ranked PDBs, log runtime/VRAM, and optionally save to Drive.

## 12. Enter your own RNA sequence

In [ ]:
JOB_NAME = "my_rna"

RNA_SEQUENCE = """
GGGAAACCCUUUGGGAAACCC
""".replace("\n","").replace(" ","").upper()

if not re.fullmatch(r"[ACGU]+", RNA_SEQUENCE):
    raise ValueError("RNA sequence must contain only A, C, G and U.")

custom_input_root = Path("/content/NuFold/custom_input")
custom_target_dir = custom_input_root / JOB_NAME
custom_target_dir.mkdir(parents=True, exist_ok=True)

custom_fasta = custom_target_dir / f"{JOB_NAME}.fasta"
custom_fasta.write_text(f">{JOB_NAME}\n{RNA_SEQUENCE}\n")

print("Target:", JOB_NAME)
print("Length:", len(RNA_SEQUENCE))
print("FASTA:", custom_fasta)

## 13. Install IPknot and generate predicted secondary structure

In [ ]:
%cd /content

ipknot = Path("/content/ipknot")

if not ipknot.exists():
    !wget -q https://github.com/satoken/ipknot/releases/download/v1.1.0/ipknot-1.1.0-x86_64-linux.zip
    !unzip -q ipknot-1.1.0-x86_64-linux.zip
    !cp ipknot-1.1.0-x86_64-linux/ipknot /content/ipknot
    !chmod +x /content/ipknot

custom_ss = custom_target_dir / f"{JOB_NAME}.single.ipknot.ss"

with custom_ss.open("w") as f:
    subprocess.run([str(ipknot), str(custom_fasta)], stdout=f, check=True)

print("Predicted SS:")
print(custom_ss.read_text())

## 14. Upload your precomputed A3M

Upload exactly one `.a3m` file for this target. For the FYP benchmark, this should come from the standardized team MSA pipeline.

In [ ]:
from google.colab import files

uploaded = files.upload()
a3m_files = [name for name in uploaded if name.lower().endswith(".a3m")]

if len(a3m_files) != 1:
    raise ValueError("Please upload exactly one .a3m file.")

custom_a3m = custom_target_dir / f"{JOB_NAME}.a3m"
shutil.move(a3m_files[0], custom_a3m)

print("A3M saved:", custom_a3m)

## 15. Validate custom input folder

In [ ]:
required_custom = [custom_fasta, custom_a3m, custom_ss]

for p in required_custom:
    print(p, "exists =", p.exists())

missing = [str(p) for p in required_custom if not p.exists()]
if missing:
    raise FileNotFoundError("Missing custom NuFold inputs: " + ", ".join(missing))

## 16. Run NuFold on the custom target

In [ ]:
%cd /content/NuFold

custom_output_root = Path("/content/NuFold/custom_output")
custom_output_root.mkdir(parents=True, exist_ok=True)

custom_gpu_samples = []
custom_stop_monitor = False

def monitor_custom_gpu():
    global custom_stop_monitor
    while not custom_stop_monitor:
        try:
            out = subprocess.check_output(
                ["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if out:
                custom_gpu_samples.append(int(out[0]))
        except Exception:
            pass
        time.sleep(0.5)

custom_thread = threading.Thread(target=monitor_custom_gpu, daemon=True)
custom_thread.start()
custom_start = time.perf_counter()

custom_proc = subprocess.run([
    "python","run_nufold.py",
    "--ckpt_path","checkpoints/global_step145245.pt",
    "--input_fasta",str(custom_fasta),
    "--input_dir",str(custom_input_root),
    "--output_dir",str(custom_output_root),
    "--config_preset","initial_training"
], cwd="/content/NuFold", capture_output=True, text=True)

custom_elapsed = time.perf_counter() - custom_start
custom_stop_monitor = True
custom_thread.join(timeout=2)
custom_peak_gpu_mb = max(custom_gpu_samples) if custom_gpu_samples else None

print("===== STDOUT =====")
print(custom_proc.stdout)
print("===== STDERR =====")
print(custom_proc.stderr)
print("=== CUSTOM RUN SUMMARY ===")
print("Exit code:", custom_proc.returncode)
print(f"Runtime: {custom_elapsed:.2f} s")
print("Peak observed GPU memory:", f"{custom_peak_gpu_mb} MiB" if custom_peak_gpu_mb is not None else "Unavailable")

if custom_proc.returncode != 0:
    raise RuntimeError("Custom NuFold inference failed. Check STDERR above.")

## 17. Verify custom ranked predictions

In [ ]:
custom_target_output = custom_output_root / JOB_NAME
custom_rank1 = custom_target_output / f"{JOB_NAME}_rank_1.pdb"

print("Rank-1 prediction:", custom_rank1)
print("Exists:", custom_rank1.exists())

for p in sorted(custom_target_output.glob("*")):
    print(p)

if not custom_rank1.exists():
    raise FileNotFoundError("Custom rank-1 PDB was not generated.")

## 18. Save custom-run reproducibility metadata

In [ ]:
custom_record = {
    "model":"NuFold",
    "model_commit":commit,
    "checkpoint":"global_step145245.pt",
    "target":JOB_NAME,
    "sequence_length":len(RNA_SEQUENCE),
    "input_fasta":str(custom_fasta),
    "input_msa":str(custom_a3m),
    "input_ss":str(custom_ss),
    "output_rank_1":str(custom_rank1),
    "python":sys.version.split()[0],
    "numpy":np.__version__,
    "pytorch":torch.__version__,
    "gpu":gpu_name,
    "runtime_seconds":round(custom_elapsed,3),
    "peak_gpu_memory_mib_observed":custom_peak_gpu_mb,
    "exit_code":custom_proc.returncode
}

custom_log_path = Path(f"/content/nufold_{JOB_NAME}_run_log.csv")
with custom_log_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=custom_record.keys())
    w.writeheader()
    w.writerow(custom_record)

print(custom_record)
print("Saved:", custom_log_path)

## 19. Optional — save custom target outputs to Google Drive

In [ ]:
# OPTIONAL CELL
# from google.colab import drive
# drive.mount("/content/drive")
# destination = Path("/content/drive/MyDrive/RNA3D_Benchmark/NuFold") / JOB_NAME
# destination.mkdir(parents=True, exist_ok=True)
# shutil.copytree(custom_target_output, destination/"predictions", dirs_exist_ok=True)
# shutil.copy2(custom_log_path, destination/custom_log_path.name)
# print("Saved to:", destination)

## 20. Benchmark policy for this FYP

Primary NuFold benchmark:
- one standardized MSA-generation policy across targets
- one standardized predicted-secondary-structure pipeline across targets
- no native 3D coordinates during inference
- no native SS in the primary benchmark
- native structure is evaluation-only
- record model commit, checkpoint, runtime, GPU and peak VRAM
- share precomputed A3M/SS files across team members where practical